In [1]:
#===================================================================
'''
1. 이진분류 실습 
- 이진분류 모델 학습 
- 학습 결과 저장 
- 저장된 모델 / 스케일러 / 학습 이력 불러오기 
- 신규 데이터 예측 
- 학습결과 및 예측결과 시각화 
'''

'\n1. 이진분류 실습 \n- 이진분류 모델 학습 \n- 학습 결과 저장 \n- 저장된 모델 / 스케일러 / 학습 이력 불러오기 \n- 신규 데이터 예측 \n- 학습결과 및 예측결과 시각화 \n'

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, auc
from sklearn.datasets import load_breast_cancer      # 유방암 진단 데이터 셋 읽어오기
import joblib                                        # 파이썬 객체 저장을 위한 joblib

import tensorflow as tf                              # 딥러닝 프레임 워크 

# 케라스의 모델 저장 및 로드 기능을 사용하기 위한 필요 모듈
from tensorflow.keras import Sequential 
from tensorflow.keras.layers import Dense, Dropout    # 레이어 구성, 드롭아웃 적용 위한 라이브러리 
from tensorflow.keras.callbacks import EarlyStopping  # 학습 조기종료 콜백 라이브러리 

np.random.seed(42)
tf.random.set_seed(42)

In [5]:
! pip install 

Package                   Version
------------------------- ------------
absl-py                   2.5.0
annotated-doc             0.0.5
annotated-types           0.8.0
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
asttokens                 3.0.1
astunparse                1.6.3
async-lru                 2.3.0
attrs                     26.1.0
babel                     2.18.0
beautifulsoup4            4.15.0
bleach                    6.4.0
branca                    0.8.2
brotlicffi                1.2.0.0
certifi                   2026.7.22
cffi                      2.1.0
charset-normalizer        3.4.7
click                     8.4.2
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.2
cycler                    0.12.1
debugpy                   1.8.21
decorator                 5.3.1
defusedxml                0.7.1
dnspython                 2.8.0
exceptiongroup            1.3.1
executing        

In [8]:
# ============================================
# 1. 데이터 읽어오기
# ============================================

data = load_breast_cancer()

X = data.data      # 입력 x data
y = data.target    # 타겟 데이터 y 데이터 (예측하고자 하는 데이터의 정답 데이터)

feature_names = data.feature_names    # 특성 이름
df = pd.DataFrame(X, columns=feature_names)    # 데이터 확인을 위해 데이터프레임으로 변환

df['target'] = y

print('전체 데이터 shape: ', df.shape)

전체 데이터 shape:  (569, 31)


In [9]:
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [12]:
#=========================================
#2. 학습용/테스트용 데이터 분할     (학습데이터 80%, 테스트데이터 20%)
#=========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [14]:
# ============================================
# 3. 스케일링
# ============================================

scaler = StandardScaler()                         # 스케일러 객체 생성
X_train_scaled = scaler.fit_transform(X_train)   # 학습 데이터 기준으로 평균과 표준편차를 학습하고 변환
X_test_scaled = scaler.transform(X_test)         # 테스트 데이터는 학습 데이터 기준으로 변환

In [15]:
# ============================================
# 4. 모델 정의
# ============================================

model = Sequential()                  # 순차형 신경망 모델 생성

# 첫 번째 은닉층 추가
model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model.add(Dropout(0.2))               # 과적합 방지를 위해 드롭아웃 20% 적용

# 두 번째 은닉층 추가
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))

#출력층 
model.add(Dense(1, activation='sigmoid')) # 이진 분류 모델(0,1)

# 모델 구조 출력
model.summary()

D:\Anaconda3\envs\ai\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │           1,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)